# Cache d'images prétraitées pour l'entraînement des CNN

**Projet ForestFireDetection** – Détection précoce de feux de forêt (fumée / feu)

Entraîner un CNN demande de relire les images à chaque époque. Le décodage des JPEG sur CPU plafonne à
~170 images/s, alors que le GPU entraîne ConvNeXt-Tiny à ~480 images/s : le GPU attendrait les images.

Ce notebook redimensionne **une seule fois** les 21 527 images en 224×224 et les enregistre dans un fichier
`uint8` (~3,2 Go, non versionné). Le prétraitement est **identique** à celui de
`preprocessing_feature_extraction.ipynb` (RGB, redimensionnement 224×224 sans recadrage, décodage JPEG réduit) ;
la normalisation ImageNet est appliquée plus tard, sur le GPU.

## 1. Imports et configuration

In [ ]:
import sys
import time
from pathlib import Path

# Ajout de la racine du projet au chemin Python pour importer src/
ROOT = Path.cwd().resolve()
sys.path.append(str(ROOT.parent if ROOT.name == "notebooks" else ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms

from src.config import (
    FEATURES_META_PATH, CACHE_DIR, IMAGE_CACHE_PATH, IMAGE_CACHE_INDEX_PATH,
    IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD, BATCH_SIZE,
)
from src.datasets import DFireDataset

CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Images à mettre en cache

Même ordre que la matrice de features (train, val, test) : `resnet50_metadata.csv`.

In [ ]:
meta = pd.read_csv(FEATURES_META_PATH)
print(f"Images : {len(meta)}")
print(meta["final_split"].value_counts().reindex(["train", "val", "test"]))

## 3. Création du cache

Transformation : redimensionnement 224×224 (sans recadrage) puis tenseur `uint8` (valeurs 0–255, 3 canaux).
Le fichier est écrit directement sur disque par blocs (`open_memmap`), sans tout charger en mémoire.

In [ ]:
to_uint8 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),     # image entière, sans recadrage
    transforms.PILToTensor(),                    # uint8 (3, H, W), valeurs 0-255
])
dataset = DFireDataset(meta, transform=to_uint8)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

cache = np.lib.format.open_memmap(IMAGE_CACHE_PATH, mode="w+", dtype=np.uint8,
                                  shape=(len(meta), 3, IMG_SIZE, IMG_SIZE))
t0, start, ids = time.time(), 0, []
for batch_idx, (images, image_ids) in enumerate(loader):
    cache[start:start + len(images)] = images.numpy()
    start += len(images)
    ids.extend(image_ids)
    if (batch_idx + 1) % 50 == 0:
        print(f"  batch {batch_idx + 1}/{len(loader)} – {time.time() - t0:.0f} s")
cache.flush()
del cache

# Index : position de chaque image dans le cache
assert ids == meta["image_id"].tolist()
index = meta[["image_id", "final_split"]].assign(position=np.arange(len(meta)))
index.to_csv(IMAGE_CACHE_INDEX_PATH, index=False)
print(f"\nCache écrit en {time.time() - t0:.0f} s : {IMAGE_CACHE_PATH.name} "
      f"({IMAGE_CACHE_PATH.stat().st_size / 1024**3:.2f} Go)")

## 4. Vérifications

1. Forme et type du cache.
2. **Équivalence avec le preprocessing de l'extraction de features** : une image du cache normalisée doit être
   identique à la même image passée par le pipeline `Resize → ToTensor → Normalize`.
3. Contrôle visuel de quelques images.

In [ ]:
cache = np.load(IMAGE_CACHE_PATH, mmap_mode="r")
print(f"Forme : {cache.shape} | type : {cache.dtype}")

reference = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
reference_ds = DFireDataset(meta, transform=reference)
mean, std = np.array(IMAGENET_MEAN).reshape(3, 1, 1), np.array(IMAGENET_STD).reshape(3, 1, 1)
rng = np.random.default_rng(0)
max_diff = 0.0
for pos in rng.choice(len(meta), size=20, replace=False):
    from_cache = (cache[pos] / 255.0 - mean) / std
    max_diff = max(max_diff, float(np.abs(from_cache - reference_ds[pos][0].numpy()).max()))
print(f"Écart maximal cache vs pipeline de référence (20 images) : {max_diff:.2e}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, pos in zip(axes.flat, rng.choice(len(meta), size=10, replace=False)):
    ax.imshow(cache[pos].transpose(1, 2, 0))
    ax.set_title(f"{meta.loc[pos, 'image_id']} ({meta.loc[pos, 'final_split']})", fontsize=8)
    ax.axis("off")
plt.suptitle("Images du cache (224×224)")
plt.tight_layout()
plt.show()

## 5. Conclusions

_Chiffres issus de l'exécution du 13/11/2026._

- **21 527 images** mises en cache en **95 s** dans `images_224.npy` : tableau `uint8` (21 527, 3, 224, 224),
  **3,02 Go**, non versionné (`data/processed/` est ignoré par git).
- **Équivalence vérifiée** avec le preprocessing de l'extraction de features : écart maximal de **3,65 × 10⁻⁷**
  sur 20 images tirées au hasard (arrondis de calcul en virgule flottante), donc images identiques.
- Gain : l'entraînement n'a plus à décoder les JPEG (~170 images/s sur CPU) ; il peut suivre le rythme du GPU
  (~480 images/s mesurées pour ConvNeXt-Tiny en entraînement).

**Suite – modèles semi-supervisés (`semi_supervised_models.ipynb`)** : entraînement des modèles A, B et C sur ce cache.